In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

## 1. Cấu hình

In [ ]:
DATASET_DIR = '/kaggle/input/datasets/e1trnnguynvkhang/medicine-dataset/dataset_resized'
MODEL_DIR   = '/kaggle/working/models'
MODEL_BEST  = f'{MODEL_DIR}/siamese_best.pth'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

IMG_SIZE        = 384
BATCH_SIZE      = 16
EPOCHS          = 25
LR_HEAD         = 1e-4   # LR khi backbone còn đóng băng
LR_BACKBONE     = 1e-5   # LR sau khi mở băng fine-tune
EMBED_DIM       = 256
VAL_RATIO       = 0.2
PAIRS_PER_CLASS = 100
NUM_WORKERS     = 2
UNFREEZE_EPOCH  = 6

## 2. Dataset
Tạo các cặp ảnh (positive / negative) cho Siamese training.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def load_dataset(root: str) -> dict:
    """Load ảnh theo class, bỏ qua class có ít hơn 2 ảnh."""
    data = {}
    for cls_dir in sorted(Path(root).iterdir()):
        if not cls_dir.is_dir():
            continue
        imgs = [str(p) for p in cls_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
        if len(imgs) >= 2:
            data[cls_dir.name] = imgs
    return data

def train_val_split(data: dict, val_ratio: float = 0.2) -> tuple[dict, dict]:
    train, val = {}, {}
    for cls, imgs in data.items():
        random.shuffle(imgs)
        n = max(1, int(len(imgs) * val_ratio))
        val[cls], train[cls] = imgs[:n], imgs[n:]
    return train, val


class SiameseDataset(Dataset):
    def __init__(self, data: dict, n_pairs: int, transform=None):
        self.data      = data
        self.classes   = list(data.keys())
        self.n_pairs   = n_pairs
        self.transform = transform
        self.pairs     = self._generate_pairs()

    def _generate_pairs(self) -> list:
        pairs = []
        # Positive pairs: 2 ảnh cùng class
        for cls, imgs in self.data.items():
            if len(imgs) < 2:
                continue
            for _ in range(PAIRS_PER_CLASS):
                a, b = random.sample(imgs, 2)
                pairs.append((a, b, 1))
        # Negative pairs: số lượng bằng positive, khác class
        for _ in range(len(pairs)):
            cls_a, cls_b = random.sample(self.classes, 2)
            a = random.choice(self.data[cls_a])
            b = random.choice(self.data[cls_b])
            pairs.append((a, b, 0))
        random.shuffle(pairs)
        return pairs

    def reshuffle(self):
        """Tạo lại cặp ảnh ngẫu nhiên mỗi epoch để tránh overfitting."""
        self.pairs = self._generate_pairs()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2, label = self.pairs[idx]
        img1 = Image.open(p1).convert('RGB')
        img2 = Image.open(p2).convert('RGB')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [ ]:
# Augmentation mạnh để model học đặc trưng hình dạng thay vì chỉ màu sắc
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    # 40% ảnh bị chuyển sang grayscale → model buộc phải học hình dạng, không chỉ màu
    transforms.RandomGrayscale(p=0.4),
    # Biến đổi màu sắc ngẫu nhiên → model học bất biến với ánh sáng/điều kiện chụp
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.4, hue=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

all_data   = load_dataset(DATASET_DIR)
train_data, val_data = train_val_split(all_data, VAL_RATIO)

train_ds = SiameseDataset(train_data, PAIRS_PER_CLASS, train_tf)
val_ds   = SiameseDataset(val_data,   50,               val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train pairs: {len(train_ds)} | Val pairs: {len(val_ds)}")

## 3. Model · Loss

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim: int = 256):
        super().__init__()
        backbone       = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features  = backbone.features
        self.pool      = nn.AdaptiveAvgPool2d(1)
        self.projector = nn.Sequential(
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.features(x)).flatten(1)
        return F.normalize(self.projector(x), p=2, dim=1)


class SiameseNetwork(nn.Module):
    def __init__(self, embed_dim: int = 256):
        super().__init__()
        self.net = EmbeddingNet(embed_dim)

    def forward(self, x1, x2):
        return self.net(x1), self.net(x2)

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ContrastiveLoss(nn.Module):
    """Kéo cặp cùng class lại gần nhau, đẩy cặp khác class ra xa ít nhất `margin`."""
    def __init__(self, margin: float = 2.0):
        super().__init__()
        self.margin = margin

    def forward(self, e1: torch.Tensor, e2: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        d = F.pairwise_distance(e1, e2)
        loss = label * d.pow(2) + (1 - label) * F.relu(self.margin - d).pow(2)
        return loss.mean()


model     = SiameseNetwork(EMBED_DIM).to(device)
criterion = ContrastiveLoss(margin=2.0)

## 4. Training loop

In [ ]:
# Đóng băng backbone — chỉ train projection head trước để ổn định embedding space
print("Đóng băng MobileNetV2 backbone...")
for param in model.net.features.parameters():
    param.requires_grad = False

optimizer     = torch.optim.Adam(model.parameters(), lr=LR_HEAD)
best_val_loss = float('inf')
history       = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):

    # Mở băng backbone từ UNFREEZE_EPOCH, dùng LR nhỏ hơn để fine-tune nhẹ
    if epoch == UNFREEZE_EPOCH:
        print(f"\n[Epoch {epoch}] Mở khóa backbone để fine-tune...")
        for param in model.net.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=LR_BACKBONE)

    train_ds.reshuffle()

    # ── Train ────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Train]', leave=False)
    for img1, img2, label in pbar:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer.zero_grad()
        e1, e2 = model(img1, img2)
        loss   = criterion(e1, e2, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    # ── Validation ───────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            e1, e2 = model(img1, img2)
            val_loss += criterion(e1, e2, label).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    saved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch, 'embed_dim': EMBED_DIM, 'model_state': model.state_dict()}, MODEL_BEST)
        saved = '  💾 Best!'

    print(f'Epoch {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}{saved}')

print(f'\nBest Val Loss: {best_val_loss:.4f} — Model lưu tại: {MODEL_BEST}')

## 5. Plot loss curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Contrastive Loss')
plt.title('Siamese Network — Loss Curve')
plt.legend()
plt.tight_layout()
plt.show()